# DataPact - A6 Part 1: Hugging Face model lab

**INFO 490 MG, Team 15. Part 1 (Route A, prompt engineering): Hriday Agarwal.**

DataPact checks recurring data files against a versioned *data contract*. When a check fails,
someone has to decide whether the **data is wrong** or the **contract is out of date**. This
notebook tests whether a small local language model can help with that decision.

**The task given to every model.** One failed validation rule goes in (dataset, column, rule,
rows failed, sample values). The model must reply with only this JSON:

```json
{"explanation": "<one plain-English sentence>",
 "likely_cause": "DATA_ISSUE" | "CONTRACT_UPDATED",
 "suggested_action": "<one short sentence>"}
```

JSON matters because the Django view that uses the model has to parse the answer with code.

**What the notebook does**

1. Installs libraries and downloads the model weights (top cells).
2. Lists 15 openly downloadable models in 5 size categories.
3. **Matrix test:** all 15 models on the same 8 fixed inputs, with the same zero-shot prompt.
4. **Route A:** the 3 best models re-tested with zero-, one-, few- and many-shot prompts.
5. **Evaluation:** structure, accuracy, speed and memory, plus a combined score.

**How to run it:** in Google Colab choose *Runtime > Change runtime type > T4 GPU*, then
*Runtime > Run all*. A full run takes roughly 30-60 minutes, mostly downloads.
Setting `SMOKE_TEST = True` below runs a 2-model, 2-input version in a couple of minutes
(this is how the notebook was checked on a laptop before the Colab run).

## 1. Install libraries

* `transformers`: Hugging Face's library that downloads a model by name and runs it.
* `accelerate`: places the model on the GPU.
* `bitsandbytes`: 4-bit loading for the Large category (CUDA GPUs only, so Colab).

PyTorch (`torch`) is already installed on Colab.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
packages = ["transformers==5.20.1", "accelerate==1.15.0", "pandas", "matplotlib"]
if IN_COLAB:
    packages.append("bitsandbytes")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Running in Colab:", IN_COLAB)

## 2. Settings and the 15 models

Models were found through the Hugging Face Open LLM Leaderboard and model hub. Every one is
**not gated** (no login or licence approval needed), is an **instruct/chat** version (trained to
follow instructions, not just continue text), and loads with plain `transformers`.

Parameter counts are the real totals from each model's `safetensors` metadata, so a name can
mislead: "Falcon3-1B" is really 1.67B. Categories were drawn where the models naturally
cluster:

| Category | Parameters |
|---|---|
| Ultra-light | < 0.5B |
| Light | 0.5B - 1.6B |
| Small | 1.6B - 3B |
| Medium | 3B - 5B |
| Large | 5B - 10B (loaded in 4-bit) |

In [ ]:
import os

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"  # set True for a quick check

MODELS = [
    # (category, Hugging Face id, parameters in billions)
    ("Ultra-light", "HuggingFaceTB/SmolLM2-135M-Instruct", 0.13),
    ("Ultra-light", "HuggingFaceTB/SmolLM2-360M-Instruct", 0.36),
    ("Ultra-light", "Qwen/Qwen2.5-0.5B-Instruct", 0.49),
    ("Light", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", 1.10),
    ("Light", "allenai/OLMo-2-0425-1B-Instruct", 1.48),
    ("Light", "Qwen/Qwen2.5-1.5B-Instruct", 1.54),
    ("Small", "tiiuae/Falcon3-1B-Instruct", 1.67),
    ("Small", "HuggingFaceTB/SmolLM2-1.7B-Instruct", 1.71),
    ("Small", "ibm-granite/granite-3.1-2b-instruct", 2.53),
    ("Medium", "Qwen/Qwen2.5-3B-Instruct", 3.09),
    ("Medium", "tiiuae/Falcon3-3B-Instruct", 3.23),
    ("Medium", "microsoft/Phi-3.5-mini-instruct", 3.82),
    ("Large", "Qwen/Qwen2.5-7B-Instruct", 7.62),
    ("Large", "tiiuae/Falcon3-7B-Instruct", 7.46),
    ("Large", "ibm-granite/granite-3.1-8b-instruct", 8.17),
]
CATEGORY_ORDER = ["Ultra-light", "Light", "Small", "Medium", "Large"]
TOP_N_FOR_ROUTE_A = 3
MAX_NEW_TOKENS = 160

if SMOKE_TEST:
    MODELS = [m for m in MODELS if m[1] in (
        "HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct")]
    TOP_N_FOR_ROUTE_A = 1

import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# 4-bit loading needs bitsandbytes, which needs an NVIDIA GPU. Without one, skip Large.
QUANTIZE_LARGE = DEVICE == "cuda"
if not QUANTIZE_LARGE:
    MODELS = [m for m in MODELS if m[0] != "Large"]

print("Device:", DEVICE, "| smoke test:", SMOKE_TEST, "| models to test:", len(MODELS))
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Download the model weights

The weights are the learned numbers inside each model, stored in `.safetensors` files.
`snapshot_download` saves them to the Hugging Face cache (outside the repository, so they can
never be committed). Downloading here, before any testing, keeps download time out of the
speed measurements.

The Large models (about 15 GB each at full precision) are **not** downloaded here: all three
together would nearly fill Colab's disk. Each one is downloaded just before its test and its
files are deleted straight after.

In [ ]:
import shutil

from huggingface_hub import scan_cache_dir, snapshot_download

WEIGHT_FILES = ["*.json", "*.safetensors", "*.model", "*.txt", "*.py", "*.tiktoken"]


def download(model_id):
    return snapshot_download(model_id, allow_patterns=WEIGHT_FILES)


def delete_from_cache(model_id):
    for repo in scan_cache_dir().repos:
        if repo.repo_id == model_id:
            hashes = [rev.commit_hash for rev in repo.revisions]
            scan_cache_dir().delete_revisions(*hashes).execute()


for category, model_id, _ in MODELS:
    if category != "Large":
        print("downloading", model_id)
        download(model_id)

total, used, free = shutil.disk_usage("/")
print(f"Done. Free disk: {free / 1e9:.1f} GB")

## 4. The fixed inputs

**Test set:** 8 failed checks written in DataPact's own vocabulary, each with the right answer.
Four are bad data (impossible values, a placeholder like `-9999`, malformed IDs, duplicates) and
four are legitimate new values the contract does not know yet (a new enrolment status, a new
library branch, a new benefit code, a new winter term).

The 4/4 split is deliberate: a model that always answers `DATA_ISSUE` scores 50%, so accuracy
above 50% means the model is reading the evidence, not guessing.

**Example pool:** 12 more labelled cases, used only as worked examples in the Route A prompts.
None of them appears in the test set, so a model cannot score by copying an answer it was shown.

In [ ]:
TEST_CASES = [
    dict(id="T1", dataset="Monthly Enrollment Export", column="credit_hours",
         rule='RANGE {"min": 0, "max": 24}', failed="47 of 4,812 rows",
         samples="[31, 99, -2]", label="DATA_ISSUE"),
    dict(id="T2", dataset="Monthly Enrollment Export", column="status",
         rule='ALLOWED_VALUES ["ENROLLED", "DROPPED", "WITHDRAWN"]', failed="4 of 4,812 rows",
         samples='["AUDIT", "AUDIT", "AUDIT", "AUDIT"]', label="CONTRACT_UPDATED"),
    dict(id="T3", dataset="Monthly Enrollment Export", column="student_id",
         rule='REGEX "^[0-9]{8}$"', failed="12 of 4,812 rows",
         samples='["1234567", "12345678A", "ABC00012"]', label="DATA_ISSUE"),
    dict(id="T4", dataset="Sensor Readings - Lab 3", column="temperature_c",
         rule='RANGE {"min": -10, "max": 50}', failed="6 of 8,640 rows",
         samples="[-9999, -9999, -9999]", label="DATA_ISSUE"),
    dict(id="T5", dataset="Library Circulation Log", column="branch",
         rule='ALLOWED_VALUES ["MAIN", "GRAINGER", "UNDERGRAD"]', failed="310 of 9,800 rows",
         samples='["FUNK_ACES", "FUNK_ACES", "FUNK_ACES"]', label="CONTRACT_UPDATED"),
    dict(id="T6", dataset="Payroll Deductions Export", column="deduction_code",
         rule='ALLOWED_VALUES ["FED", "STATE", "MEDICARE", "SOCSEC"]', failed="512 of 3,402 rows",
         samples='["HSA", "HSA", "HSA"]', label="CONTRACT_UPDATED"),
    dict(id="T7", dataset="Course Catalog Snapshot", column="term_code",
         rule='ALLOWED_VALUES ["FA", "SP", "SU"]', failed="85 of 2,100 rows",
         samples='["WI", "WI", "WI"]', label="CONTRACT_UPDATED"),
    dict(id="T8", dataset="Facilities Work Orders", column="order_id",
         rule="UNIQUE", failed="14 of 2,210 rows",
         samples='["WO-10233", "WO-10233", "WO-10871", "WO-10871"]', label="DATA_ISSUE"),
]

EXAMPLE_POOL = [
    (dict(dataset="Weekly Sales Extract", column="quantity", rule='RANGE {"min": 1, "max": 1000}',
          failed="9 of 13,120 rows", samples="[0, -3, 0]"),
     dict(explanation="Some sales rows have a quantity of zero or below, which cannot happen for a real sale.",
          likely_cause="DATA_ISSUE", suggested_action="Ask the sales team to correct the quantities in those rows.")),
    (dict(dataset="Monthly Enrollment Export", column="college", rule='ALLOWED_VALUES ["LAS", "ENG", "BUS", "ACES"]',
          failed="220 of 4,812 rows", samples='["CDS", "CDS", "CDS"]'),
     dict(explanation="Many students are listed under CDS, which looks like a newly created college rather than a typo.",
          likely_cause="CONTRACT_UPDATED", suggested_action="Confirm CDS is a real college and add it to the allowed values.")),
    (dict(dataset="Sensor Readings - Lab 3", column="humidity_pct", rule='TYPE_MATCH {"type": "number"}',
          failed="4 of 8,640 rows", samples='["N/A", "err", "N/A"]'),
     dict(explanation="A few humidity readings are text error markers instead of numbers.",
          likely_cause="DATA_ISSUE", suggested_action="Check the sensor logger and re-export the affected readings.")),
    (dict(dataset="Facilities Work Orders", column="priority", rule='ALLOWED_VALUES ["LOW", "MEDIUM", "HIGH"]',
          failed="140 of 2,210 rows", samples='["URGENT", "URGENT", "URGENT"]'),
     dict(explanation="Work orders now use an URGENT priority that the contract does not list yet.",
          likely_cause="CONTRACT_UPDATED", suggested_action="Confirm URGENT is an official priority and add it to the contract.")),
    (dict(dataset="Library Circulation Log", column="checkout_date", rule="NOT_NULL",
          failed="2 of 9,800 rows", samples="[null, null]"),
     dict(explanation="Two checkouts are missing their date, which every checkout must have.",
          likely_cause="DATA_ISSUE", suggested_action="Find the missing dates in the circulation system and fix the export.")),
    (dict(dataset="Course Catalog Snapshot", column="credit_hours", rule='RANGE {"min": 1, "max": 5}',
          failed="18 of 2,100 rows", samples="[6, 6, 6]"),
     dict(explanation="Several courses are worth 6 credits, consistent with a newly approved capstone format.",
          likely_cause="CONTRACT_UPDATED", suggested_action="Confirm the 6-credit courses and raise the maximum to 6.")),
    (dict(dataset="Payroll Deductions Export", column="employee_id", rule="UNIQUE",
          failed="5 of 3,402 rows", samples='["E1001", "E1001", "E2044"]'),
     dict(explanation="Some employee IDs appear more than once, so the same people were exported twice.",
          likely_cause="DATA_ISSUE", suggested_action="Remove the duplicate rows and check why the export repeated them.")),
    (dict(dataset="Weekly Sales Extract", column="region", rule='ALLOWED_VALUES ["NORTH", "SOUTH", "EAST", "WEST"]',
          failed="1,500 of 13,120 rows", samples='["ONLINE", "ONLINE", "ONLINE"]'),
     dict(explanation="A large, consistent block of sales is tagged ONLINE, which looks like a new sales channel.",
          likely_cause="CONTRACT_UPDATED", suggested_action="Confirm ONLINE is a real region and add it to the contract.")),
    (dict(dataset="Monthly Enrollment Export", column="email", rule='REGEX "^[^@]+@illinois\\.edu$"',
          failed="7 of 4,812 rows", samples='["jdoe@illinois", "@illinois.edu"]'),
     dict(explanation="A few email addresses are incomplete and would not work.",
          likely_cause="DATA_ISSUE", suggested_action="Correct the malformed addresses at the source.")),
    (dict(dataset="Sensor Readings - Lab 3", column="sensor_id", rule='ALLOWED_VALUES ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"]',
          failed="720 of 8,640 rows", samples='["S9", "S9", "S9"]'),
     dict(explanation="Readings arrive steadily from a sensor S9, which suggests a new sensor was installed.",
          likely_cause="CONTRACT_UPDATED", suggested_action="Confirm sensor S9 exists and add it to the allowed list.")),
    (dict(dataset="Facilities Work Orders", column="cost_usd", rule='TYPE_MATCH {"type": "number"}',
          failed="3 of 2,210 rows", samples='["$1,200", "TBD"]'),
     dict(explanation="A few costs were typed as text with symbols or placeholders instead of numbers.",
          likely_cause="DATA_ISSUE", suggested_action="Re-enter those costs as plain numbers.")),
    (dict(dataset="Library Circulation Log", column="item_type", rule='ALLOWED_VALUES ["BOOK", "DVD", "JOURNAL"]',
          failed="410 of 9,800 rows", samples='["LAPTOP", "LAPTOP", "LAPTOP"]'),
     dict(explanation="Many loans are for laptops, which matches a new lending service the contract predates.",
          likely_cause="CONTRACT_UPDATED", suggested_action="Confirm laptop lending and add LAPTOP to the allowed values.")),
]

SHOT_COUNTS = {"zero-shot": 0, "one-shot": 1, "few-shot": 3, "many-shot": 12}

if SMOKE_TEST:
    TEST_CASES = [TEST_CASES[0], TEST_CASES[1]]

print(len(TEST_CASES), "test cases,", len(EXAMPLE_POOL), "pool examples")

## 5. Building the prompt

Chat models expect a list of messages (`user`, `assistant`) that each model's **chat template**
turns into its own special format. The examples for one/few/many-shot are added as earlier
user-assistant turns, so the model sees "question, ideal answer" pairs before the real question.

The instructions go in the user message, not a system message, because not every model's
template supports a system role.

In [ ]:
import json

INSTRUCTIONS = (
    "You review failed data-quality checks for DataPact. A failure means either the incoming "
    "data is wrong (DATA_ISSUE) or the data is legitimate and the rule is out of date "
    "(CONTRACT_UPDATED).\n"
    "Reply with ONLY a JSON object, no other text, with exactly these keys:\n"
    '  "explanation": one plain-English sentence,\n'
    '  "likely_cause": "DATA_ISSUE" or "CONTRACT_UPDATED",\n'
    '  "suggested_action": one short sentence.'
)


def describe(case):
    return (
        f"Dataset: {case['dataset']}\nColumn: {case['column']}\nRule: {case['rule']}\n"
        f"Rows failed: {case['failed']}\nSample values: {case['samples']}"
    )


def build_messages(case, n_shots):
    messages = []
    for i, (example, answer) in enumerate(EXAMPLE_POOL[:n_shots]):
        text = describe(example)
        if i == 0:
            text = INSTRUCTIONS + "\n\n" + text
        messages.append({"role": "user", "content": text})
        messages.append({"role": "assistant", "content": json.dumps(answer)})
    question = describe(case)
    if n_shots == 0:
        question = INSTRUCTIONS + "\n\n" + question
    messages.append({"role": "user", "content": question})
    return messages


print("Zero-shot prompt for T1:\n")
print(build_messages(TEST_CASES[0], 0)[-1]["content"])

## 6. Loading, running and scoring a model

* **Loading.** Up to Medium, models load in 16-bit floats (2 bytes per parameter). Large models
  load in **4-bit** (`BitsAndBytesConfig`): about 4x smaller, so a 7B model fits on a free T4,
  at some cost in quality. Results for Large are marked with this caveat.
* **Generating.** Greedy decoding (`do_sample=False`): the model always picks its single most
  likely next token, so a rerun gives the same answer. That makes the comparison repeatable.
* **Scoring.** The first `{...}` in the reply is parsed as JSON, then checked for the three keys
  and an allowed `likely_cause`, then compared with the correct answer.
* **Memory** is the peak GPU memory used while the model answered (CUDA only).

In [ ]:
import gc
import re
import time

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

REQUIRED_KEYS = {"explanation", "likely_cause", "suggested_action"}
CAUSES = {"DATA_ISSUE", "CONTRACT_UPDATED"}


def load(model_id, category):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"dtype": torch.float16 if DEVICE != "cpu" else torch.float32}
    if category == "Large":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if category != "Large":
        model.to(DEVICE)
    model.eval()
    return tokenizer, model


def unload(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


def generate(tokenizer, model, messages):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text, seconds, len(new_tokens), inputs["input_ids"].shape[1]


def score(text, label):
    result = {"valid_json": False, "schema_ok": False, "correct": False, "predicted": None}
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return result
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return result
    if not isinstance(data, dict):
        return result
    result["valid_json"] = True
    cause = str(data.get("likely_cause", "")).strip().upper()
    result["predicted"] = cause or None
    result["schema_ok"] = REQUIRED_KEYS.issubset(data) and cause in CAUSES
    result["correct"] = cause == label
    return result


def run(model_id, category, n_shots, method):
    tokenizer, model = load(model_id, category)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    rows = []
    for case in TEST_CASES:
        text, seconds, n_new, n_prompt = generate(tokenizer, model, build_messages(case, n_shots))
        rows.append({
            "model": model_id.split("/")[-1], "model_id": model_id, "category": category,
            "method": method, "case": case["id"], "label": case["label"],
            "prompt_tokens": n_prompt, "new_tokens": n_new, "seconds": round(seconds, 2),
            "tokens_per_s": round(n_new / seconds, 1) if seconds else None,
            "output": text, **score(text, case["label"]),
        })
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None
    for row in rows:
        row["peak_mem_gb"] = round(peak_gb, 2) if peak_gb else None
    unload(model)
    return rows

## 7. Matrix test: 15 models x 8 inputs, zero-shot

Every model gets exactly the same zero-shot prompt, so differences come from the model alone.
Each model is loaded, run on all 8 inputs, then unloaded before the next one, so only one model
is in memory at a time. Large models are downloaded right before their test and deleted after.

In [ ]:
import pandas as pd

os.makedirs("results", exist_ok=True)
SIZE = {model_id: params for _, model_id, params in MODELS}

matrix_rows = []
for category, model_id, params in MODELS:
    print(f"[{category}] {model_id} ...", flush=True)
    if category == "Large":
        download(model_id)
    try:
        matrix_rows += run(model_id, category, 0, "zero-shot")
    except Exception as error:  # one broken model should not stop the other 14
        print("   failed:", type(error).__name__, error)
    if category == "Large":
        delete_from_cache(model_id)

matrix = pd.DataFrame(matrix_rows)
matrix["params_b"] = matrix["model_id"].map(SIZE)
matrix.to_csv("results/matrix_results.csv", index=False)
print("Saved results/matrix_results.csv with", len(matrix), "rows")

### Matrix results by model

* `json_%`: replies a program could parse. `schema_%`: right keys and an allowed cause.
* `accuracy_%`: `likely_cause` matched the right answer (50% = guessing).
* `score`: combined ranking used to pick the Route A models -
  **40% schema + 40% accuracy + 20% speed**, speed relative to the fastest model.
  Structure counts as much as accuracy because the app cannot use an answer it cannot parse.

In [ ]:
def summarise(frame, by):
    summary = frame.groupby(by, sort=False).agg(
        json_pct=("valid_json", "mean"), schema_pct=("schema_ok", "mean"),
        accuracy_pct=("correct", "mean"), sec_per_answer=("seconds", "mean"),
        tokens_per_s=("tokens_per_s", "mean"), peak_mem_gb=("peak_mem_gb", "max"),
    )
    for col in ("json_pct", "schema_pct", "accuracy_pct"):
        summary[col] = (summary[col] * 100).round(0)
    summary["sec_per_answer"] = summary["sec_per_answer"].round(2)
    summary["tokens_per_s"] = summary["tokens_per_s"].round(1)
    speed = summary["tokens_per_s"] / summary["tokens_per_s"].max()
    summary["score"] = (0.4 * summary["schema_pct"] + 0.4 * summary["accuracy_pct"]
                        + 20 * speed).round(1)
    return summary


by_model = summarise(matrix, ["category", "model", "model_id"]).reset_index()
by_model["params_b"] = by_model["model_id"].map(SIZE)
by_model.drop(columns="model_id").sort_values("score", ascending=False)

In [ ]:
by_category = summarise(matrix, ["category"]).reindex(
    [c for c in CATEGORY_ORDER if c in set(matrix["category"])])
by_category

In [ ]:
import matplotlib.pyplot as plt

plot = by_model.set_index("model").reindex(
    by_model.sort_values("params_b")["model"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot[["schema_pct", "accuracy_pct"]].plot.bar(ax=axes[0], color=["#2f5bd7", "#1d6b36"])
axes[0].set_title("Zero-shot: valid structure and accuracy per model")
axes[0].set_ylabel("% of the 8 test inputs")
axes[0].axhline(50, color="grey", linestyle="--", linewidth=1, label="guessing (50%)")
axes[0].legend()
axes[0].set_xlabel("model (smallest to largest)")
plot["tokens_per_s"].plot.bar(ax=axes[1], color="#855c11")
axes[1].set_title("Generation speed per model")
axes[1].set_ylabel("tokens per second")
axes[1].set_xlabel("model (smallest to largest)")
plt.tight_layout()
plt.savefig("results/matrix_chart.png", dpi=120)
plt.show()

## 8. Route A: zero-, one-, few- and many-shot prompting

The top models from the matrix test are re-run with 0, 1, 3 and 12 worked examples in front of
the real question. Same 8 test inputs, same scoring. The question: **which prompting method
gives the most reliable structure (and correct answers) for DataPact?**

More examples usually teach the format better but make the prompt longer, which costs time on
every request. `prompt_tokens` shows how much longer.

In [ ]:
top_models = (by_model.sort_values("score", ascending=False)
              .head(TOP_N_FOR_ROUTE_A)[["category", "model_id"]].values.tolist())
print("Route A models:", [m for _, m in top_models])

route_rows = []
for category, model_id in top_models:
    if category == "Large":
        download(model_id)
    for method, n_shots in SHOT_COUNTS.items():
        print(f"{model_id} - {method}", flush=True)
        route_rows += run(model_id, category, n_shots, method)
    if category == "Large":
        delete_from_cache(model_id)

route = pd.DataFrame(route_rows)
route.to_csv("results/prompting_results.csv", index=False)
print("Saved results/prompting_results.csv with", len(route), "rows")

In [ ]:
by_method = summarise(route, ["model", "method"]).drop(columns="score")
by_method["prompt_tokens"] = route.groupby(["model", "method"], sort=False)["prompt_tokens"].mean().round(0)
by_method

In [ ]:
methods = list(SHOT_COUNTS)
pivot = (route.groupby(["method", "model"])["schema_ok"].mean().unstack() * 100).reindex(methods)
ax = pivot.plot(marker="o", figsize=(8, 5))
ax.set_title("Valid structure by prompting method")
ax.set_ylabel("% of replies with a valid schema")
ax.set_xlabel("prompting method")
ax.set_ylim(-5, 105)
plt.tight_layout()
plt.savefig("results/prompting_chart.png", dpi=120)
plt.show()

## 9. Evaluation

Criteria the team uses to compare models, in order of importance for DataPact:

1. **Instruction adherence (structure).** Does the reply parse as JSON with the right keys? The
   Django view cannot use anything else.
2. **Reasoning (accuracy).** Does it tell bad data from an outdated contract? 50% is guessing.
3. **Speed.** Seconds per answer and tokens per second, since a user waits for it.
4. **Footprint.** Parameters and peak memory, which decide what server it needs.

The cell below states the results from the numbers above, so the summary always matches the run.

In [ ]:
best_overall = by_model.sort_values("score", ascending=False).iloc[0]
best_per_category = (by_model.sort_values("score", ascending=False)
                     .groupby("category").head(1).set_index("category")
                     .reindex([c for c in CATEGORY_ORDER if c in set(by_model["category"])]))
method_rank = summarise(route, ["method"]).sort_values(
    ["schema_pct", "accuracy_pct"], ascending=False)

print(f"Best model overall: {best_overall['model']} ({best_overall['params_b']}B, "
      f"{best_overall['category']}) - schema {best_overall['schema_pct']:.0f}%, "
      f"accuracy {best_overall['accuracy_pct']:.0f}%, {best_overall['sec_per_answer']}s per answer")
print("\nBest model in each category:")
for category, row in best_per_category.iterrows():
    print(f"  {category:12} {row['model']:34} schema {row['schema_pct']:>3.0f}%  "
          f"accuracy {row['accuracy_pct']:>3.0f}%  {row['sec_per_answer']}s")
print("\nPrompting methods (averaged over the Route A models), best first:")
for method, row in method_rank.iterrows():
    print(f"  {method:10} schema {row['schema_pct']:>3.0f}%  accuracy {row['accuracy_pct']:>3.0f}%  "
          f"{row['sec_per_answer']}s per answer")

In [ ]:
# A few raw replies, to show what "valid" and "invalid" look like in practice.
sample = matrix.sort_values(["schema_ok", "params_b"]).groupby("schema_ok").head(3)
for _, row in sample.iterrows():
    print(f"--- {row['model']} on {row['case']} (schema ok: {row['schema_ok']}) ---")
    print(row["output"][:400], "\n")

## 10. Findings

*Written after the full Colab run, from the tables above.*

- **Size sweet spot:** _to fill in_
- **Was the largest model clearly better?** _to fill in_
- **Most reliable prompting method:** _to fill in_
- **Recommended model for the Django integration:** _to fill in_

## 11. Download the results (Colab only)

Zips the `results/` folder (two CSVs and two charts) and downloads it, so the files can be
added to `llm-test/results/` in the repository. Then save this notebook with
*File > Download > Download .ipynb* so its outputs are kept.

In [ ]:
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("results", "zip", "results")
    files.download("results.zip")
else:
    print("Not in Colab - results are already in ./results")